<div align="center">

# Patra Toolkit: Model Card & Datasheet Demo

</div>

This notebook is a focused, end-to-end walkthrough of the core Patra Toolkit operations against the Postgres-backed Patra server:

1. Model Card creation
2. Datasheet creation
3. Model Card submission
4. Datasheet submission
5. Model Card listing
6. Datasheet listing
7. Model Card retrieval
8. Datasheet retrieval

It uses placeholder metadata rather than training a real model, so it can be run end-to-end quickly. For a fuller example that trains a real model and runs fairness/explainability scanners, see `GettingStarted.ipynb`.

In [ ]:
!pip install patra-toolkit

In [ ]:
from patra_toolkit import ModelCard, AIModel, Datasheet

## Configuration

Set the base URL of your Patra server. If you're running the backend locally via `docker compose -f docker-compose.backend.yml up --build` in `patra-knowledge-base`, this defaults to `http://localhost:8000`.

In [ ]:
patra_server_url = "http://localhost:8000"

## 1. Model Card Creation

Only `name` is required on `ModelCard` and `AIModel` -- every other field is optional. Attach an `AIModel` to describe the underlying model.

In [ ]:
mc = ModelCard(
    name="Demo_Classification_Model",
    version="1.0",
    short_description="A placeholder model card for demonstrating the Patra Toolkit API.",
    full_description=(
        "This model card does not describe a real trained model -- it exists purely to demonstrate "
        "model card creation, submission, listing, and retrieval."
    ),
    keywords="demo, patra, toolkit",
    author="Demo Author",
    input_type="Tabular",
    category="classification",
)

ai_model = AIModel(
    name="DemoModel",
    version="1.0",
    description="Placeholder AI model metadata.",
    owner="Demo Author",
    framework="sklearn",
    model_type="random_forest",
    test_accuracy=0.87,
)
ai_model.add_metric("Precision", 0.85)
ai_model.add_metric("Recall", 0.83)

mc.ai_model = ai_model
mc.validate()

## 2. Datasheet Creation

Datasheets describe datasets using DataCite-style metadata. No fields are required to construct one, but `validate()`/`submit()` require at least one title and one creator.

In [ ]:
ds = Datasheet(publication_year=2025, version="1.0")
ds.add_title("Demo Dataset")
ds.add_creator("Demo Author")
ds.add_description("A placeholder dataset for demonstrating the Patra Toolkit API.", "Abstract")

ds.validate()

## [Optional] TAPIS Authentication

Patra servers hosted as TAPIS pods require authentication using a JWT for secure access. To generate this token, authenticate with your TACC credentials. If you do not already have a TACC account, you can create one at [https://accounts.tacc.utexas.edu/begin](https://accounts.tacc.utexas.edu/begin). If your Patra server doesn't require authentication, skip this cell and pass `token=None` when submitting.

In [ ]:
tapis_token = mc.authenticate(username="<your_tacc_username>", password="<your_tacc_password>")
# tapis_token = None  # uncomment if your Patra server doesn't require authentication

## 3. Model Card Submission

In [ ]:
mc_result = mc.submit(patra_server_url=patra_server_url, token=tapis_token)
print(mc_result)
print("Model Card uuid:", mc.uuid)

## 4. Datasheet Submission

In [ ]:
ds_result = ds.submit(patra_server_url=patra_server_url, token=tapis_token)
print(ds_result)
print("Datasheet uuid:", ds.uuid)

## 5. Model Card Listing

Returns summaries (`uuid`, `name`, and other summary fields) for model cards on the server.

In [ ]:
ModelCard.list_model_cards(server_url=patra_server_url, token=tapis_token, q="Demo_Classification_Model")

## 6. Datasheet Listing

Returns summaries (`uuid`, `title`, and other summary fields) for datasheets on the server.

In [ ]:
Datasheet.list_datasheets(server_url=patra_server_url, token=tapis_token, q="Demo Dataset")

## 7. Model Card Retrieval

Retrieves the full record for a single model card by `uuid`, including its nested `ai_model` details.

In [ ]:
ModelCard.get_model_card(server_url=patra_server_url, uuid=mc.uuid, token=tapis_token)

## 8. Datasheet Retrieval

Retrieves the full DataCite-style record for a single datasheet by `uuid`.

In [ ]:
Datasheet.get_datasheet(server_url=patra_server_url, uuid=ds.uuid, token=tapis_token)

## Summary

By following this notebook, you have:
1. Created a Model Card and a Datasheet
2. [Optionally] Authenticated with TAPIS to obtain a token
3. Submitted both the Model Card and the Datasheet to a Patra server
4. Listed model cards and datasheets on the server
5. Retrieved a single model card and a single datasheet by `uuid`